# 06 — SPARQL A-Box Expert Matching (ontology-driven)

> **Companion to notebook 01, not a replacement.** `01_matching_baseline_afcp.ipynb`
> is the deliberately ontology-free *floor* (TF-IDF + rule gate). This notebook
> answers the same question — *given a 소부장 (소재·부품·장비) SME's technical
> problem, who are the experts that hold the required technology?* — but does it
> by **SPARQL traversal of the knowledge graph** instead of lexical similarity.
> Both run over the *same* ground truth and the *same* metrics so the numbers are
> directly comparable.

**The retrieval is architectural, not textual.** A problem is matched to experts
through ontology edges:

```
Problem ─requiresSkill─────────────────────→ Skill  ←hasSkill────── Expert
Problem ─involvesProcess→ Process ─requiresSkill→ Skill  ←hasSkill────── Expert
Problem ─involvesProcess──────────────────→ Process ←hasProcessExpertise─ Expert
Problem ─exhibitsFailureMode→ FM ─isDueTo→ RC ─mitigatedBy→ Mit ─mitigationProvidesSkill→ Skill
```

**Inputs**
- `ontology/sdkb-core-data.ttl` — domain KG (`make convert`); 198 nodes / 268 edges
- `ontology/sdkb-abox-experts-problems.ttl` — 110 experts + 226 SME problems lifted
  to instances linked to the same node URIs (`make abox`; see
  `scripts/build_abox_experts_problems.py` + `mappings/abox_term_aliases.json`)
- `data/experts/curated_ratings.parquet` — 7,800 3-rater ratings (0–3 Likert),
  the **identical** ground truth notebook 01 uses
- `data/problems_external/sme_problems_v1.json` — SME problem text/metadata
- `data/reports/abox_linking_report.json` — honest coverage of the lossy lift

**Run order**

```bash
make venv                         # project .venv (python3.11)
make abox PYTHON=.venv/bin/python # convert + build the A-Box TTL
make curated-ratings PYTHON=.venv/bin/python  # ground-truth parquet
.venv/bin/jupyter nbconvert --to notebook --execute --inplace \
    notebooks/06_sparql_abox_matching.ipynb
```

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rdflib import URIRef

ROOT = Path.cwd().resolve()
if not (ROOT / 'ontology' / 'sdkb-abox-experts-problems.ttl').exists() \\
        and (ROOT.parent / 'ontology' / 'sdkb-abox-experts-problems.ttl').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
import sdkb_nb as S   # single source: graph loader + bridge + metric suite

RATINGS  = ROOT / 'data' / 'experts' / 'curated_ratings.parquet'
SME_JSON = ROOT / 'data' / 'problems_external' / 'sme_problems_v1.json'
REPORT   = ROOT / 'data' / 'reports' / 'abox_linking_report.json'

if not (ROOT / 'ontology' / 'sdkb-abox-experts-problems.ttl').exists():
    raise SystemExit('Run `make abox PYTHON=.venv/bin/python` first '
                      '(needs sdkb-core-data.ttl + sdkb-abox-experts-problems.ttl).')
if not RATINGS.exists():
    raise SystemExit('Run `make curated-ratings PYTHON=.venv/bin/python` first '
                      '(ground-truth parquet).')

DATA, ONT, PFX = S.DATA, S.ONT, S.PFX
# core-data + experts/problems A-Box only (patents excluded so metrics stay
# identical to the pre-refactor run — this is the parity contract).
g = S.load_graph(ROOT, parts=('experts_problems',))

ratings = pd.read_parquet(RATINGS)
sme = json.loads(SME_JSON.read_text())
sme_problems = sme if isinstance(sme, list) else sme.get('problems', sme)
by_pid = {p['problem_id']: p for p in sme_problems}
link_report = json.loads(REPORT.read_text()) if REPORT.exists() else {}

# Same problem universe as notebook 01: SME problems that carry >=1 rating.
rated_ids = set(ratings['problem_id'].unique())
problems = [p for p in sme_problems if p.get('problem_id') in rated_ids]

# Same expert universe / ordering as 01 (deterministic tail-padding later).
expert_label = {str(r.e).rsplit('/', 1)[-1]: str(r.l)
                for r in g.query(S.PFX + 'SELECT ?e ?l WHERE '
                                 '{ ?e a ont:Expert ; rdfs:label ?l }')}
expert_ids = sorted(expert_label)

print(f'merged graph:     {len(g):,} triples (core + experts/problems A-Box)')
print(f'experts:          {len(expert_ids)}')
print(f'problems (rated): {len(problems)} of {len(sme_problems)} in sme_problems_v1.json')
print(f'ratings:          {ratings.shape}')
tr = link_report.get('term_resolution', {})
orph = link_report.get('orphans', {})
print(f'lift coverage:    {tr.get("distinct_matched")} matched / '
      f'{tr.get("distinct_unmatched")} unmatched distinct terms; '
      f'orphans: {len(orph.get("experts_with_no_ontology_link", []))} experts, '
      f'{len(orph.get("problems_with_no_ontology_link", []))} problems')

## 1. 데이터가 온톨로지에 어떻게 연결되는가 — A-Box 브리지

매칭 결과를 보기 **전에**, 소부장 문제와 전문가가 우리가 만든 온톨로지(`data/semiconductor_v0_3.json` → `sdkb-core-data.ttl`)에 *어떻게* 연결되는지를 먼저 코드로 보여준다. 연결은 `scripts/build_abox_experts_problems.py`의 결정적 2-tier 브리지로 만들어진다:

- **Tier-1 lexicon** — 자유 텍스트 태그를 노드의 `canonical_name` / id / 동의어(한국어 포함)와 정규화 매칭
- **Tier-2 alias** — `mappings/abox_term_aliases.json`의 큐레이션 별칭 (Tier-1이 놓치는 도메인어)
- 매칭된 노드는 그 **노드 타입**에 따라 A-Box 술어로 라우팅 (예: `Skill→ont:hasSkill`, `Process→ont:involvesProcess`)
- 매칭 실패 태그(DRAM·FEOL 등 온톨로지 밖)는 **버려지고 리포트에 정직하게 기록**

아래 두 셀은 빌더 로직을 그대로 import 해서(단일 소스) `원문 태그 → 온톨로지 노드 → A-Box 술어`를 표로 보여주고, 실제 TTL 트리플로 그 결과를 재확인한다.

In [ ]:
# --- §1a  소부장 문제 ↔ 온토로지 연계 (공용 브리지: scripts/sdkb_nb.py) ---
br = S.make_bridge(ROOT)   # Tier-1 lexicon + Tier-2 alias, single source

SOOBUJANG = {'materials', 'parts', 'equipment'}
prob = next(p for p in problems
            if p.get('company_type') in SOOBUJANG
            and p['problem_id'].startswith('PROB_'))
pid = prob['problem_id']

print(f'소부장 문제 {pid}  [{prob.get("company_type")} SME]  '
      f'"{prob.get("problem_title")}"\n')
print('① 원문 자유 텍스트 태그 (sme_problems_v1.json):')
for f in br.PROBLEM_FIELDS:
    v = prob.get(f)
    if v:
        print(f'   {f:20s}= {v}')

print('\n② 브리지가 만든  태그 → 온토로지 노드 → A-Box 술어:')
bt = br.bridge_table(prob, br.PROBLEM_FIELDS, 'problem')
print(bt.to_string(index=False))

print('\n③ 실제 A-Box TTL 트리플로 재확인 (problem → ontology):')
q = ('PREFIX ont: <%s>\n'
     'SELECT ?pred ?obj WHERE { ?prob ?p ?obj .\n'
     '  BIND(REPLACE(STR(?p), "%s", "ont:") AS ?pred)\n'
     '  FILTER(STRSTARTS(STR(?p), "%s")) }' % (ONT, ONT, ONT))
for r in sorted(g.query(q, initBindings={'prob': URIRef(f'{DATA}problem/{pid}')}),
                key=lambda r: str(r.pred)):
    o = str(r.obj)
    o = o.replace(DATA, '').replace('/', ':', 1) if o.startswith(DATA) else f'"{o}"'
    print(f'   data:problem/{pid}  {r.pred}  {o}')

In [ ]:
# --- §1b  전문가 ↔ 온토로지 연계 (같은 공용 브리지) ---
exp_raw = {e['expert_id']: e
           for e in json.loads(
               (ROOT / 'data' / 'experts' / 'curated_profiles_kr.json').read_text()
           )['experts']}

eid = next(e for e in expert_ids if e in exp_raw)   # EXP_001
ex = exp_raw[eid]

print(f'전문가 {eid}  {ex.get("name","")}  '
      f'({ex.get("former_employer","")}, {ex.get("years_experience","?")}y)\n')
print('① 원문 자유 텍스트 태그 (curated_profiles_kr.json):')
for f in br.EXPERT_FIELDS:
    v = ex.get(f)
    if v:
        print(f'   {f:24s}= {v}')

print('\n② 브리지가 만든  태그 → 온토로지 노드 → A-Box 술어:')
be = br.bridge_table(ex, br.EXPERT_FIELDS, 'expert')
print(be.to_string(index=False))

print('\n③ 실제 A-Box TTL 트리플로 재확인 (expert → ontology):')
for r in sorted(g.query(q, initBindings={'prob': URIRef(f'{DATA}expert/{eid}')}),
                key=lambda r: str(r.pred)):
    o = str(r.obj)
    o = o.replace(DATA, '').replace('/', ':', 1) if o.startswith(DATA) else f'"{o}"'
    print(f'   data:expert/{eid}  {r.pred}  {o}')

print('\n→ 이렇게 문제·전문가가 *같은* 온토로지 노드 URI를 가리키므로, '
      '다음 절의 단일 SPARQL이 그 노드를 타고 둘을 잇는다.')

## 2. The SPARQL retrieval model

`MATCH_QUERY` is the whole matcher. For a bound `?prob` it collects every
ontology *need* node reachable by the four edge patterns above, then keeps
experts whose `hasSkill` / `hasProcessExpertise` set intersects those needs.
The **score is the count of distinct overlapping need-nodes** — a pure graph
structural signal, no text, no embeddings, no TF-IDF.

`rank_experts(pid)` turns that into a full 110-long ranking (overlap desc,
`expert_id` as the deterministic tie-break) and **pads the tail with the
remaining zero-overlap experts** so every problem yields a complete ranking —
exactly the shape notebook 01 produces, which is what makes the metrics in §3
comparable rather than merely adjacent.

In [4]:
PFX = f'''PREFIX ont:  <{ONT}>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
'''

# problem -> required ontology 'need' nodes (skills + processes), 4 edge paths.
MATCH_QUERY = PFX + '''
SELECT ?e (COUNT(DISTINCT ?need) AS ?overlap) WHERE {
  { ?prob ont:requiresSkill ?need }
  UNION { ?prob ont:involvesProcess ?p . ?p ont:requiresSkill ?need }
  UNION { ?prob ont:involvesProcess ?need }
  UNION { ?prob ont:exhibitsFailureMode ?fm . ?fm ont:isDueTo ?rc .
          ?rc ont:mitigatedBy ?m . ?m ont:mitigationProvidesSkill ?need }
  ?e a ont:Expert .
  { ?e ont:hasSkill ?need } UNION { ?e ont:hasProcessExpertise ?need }
} GROUP BY ?e ORDER BY DESC(?overlap)'''

# Same need-set, but list the nodes (for the demo / introspection).
NEEDS_QUERY = PFX + '''
SELECT DISTINCT ?need ?lab WHERE {
  { ?prob ont:requiresSkill ?need }
  UNION { ?prob ont:involvesProcess ?p . ?p ont:requiresSkill ?need }
  UNION { ?prob ont:involvesProcess ?need }
  UNION { ?prob ont:exhibitsFailureMode ?fm . ?fm ont:isDueTo ?rc .
          ?rc ont:mitigatedBy ?m . ?m ont:mitigationProvidesSkill ?need }
  OPTIONAL { ?need <http://www.w3.org/2004/02/skos/core#prefLabel> ?lab }
}'''


def _uri(pid: str) -> URIRef:
    return URIRef(f'{DATA}problem/{pid}')


def problem_needs(pid: str) -> list[tuple[str, str]]:
    """[(node_id, label)] the ontology requires for this problem."""
    out = []
    for r in g.query(NEEDS_QUERY, initBindings={'prob': _uri(pid)}):
        nid = str(r.need).replace(DATA, '').replace('/', ':')
        out.append((nid, str(r.lab) if r.lab else nid))
    return out


def rank_experts(pid: str) -> list[str]:
    """Full 110-long ranking: graph overlap desc, expert_id tie-break,
    zero-overlap experts appended in deterministic order (parity with 01)."""
    scored: dict[str, int] = {}
    for r in g.query(MATCH_QUERY, initBindings={'prob': _uri(pid)}):
        scored[str(r.e).rsplit('/', 1)[-1]] = int(r.overlap)
    ranked = sorted(scored, key=lambda e: (-scored[e], e))
    tail = [e for e in expert_ids if e not in scored]
    return ranked + tail


# smoke check
_demo_pid = next(p['problem_id'] for p in problems
                 if p['problem_id'].startswith('PROB_'))
print(f'{_demo_pid}: {len(problem_needs(_demo_pid))} need-nodes, '
      f'top expert = {rank_experts(_demo_pid)[0]}')

PROB_001: 3 need-nodes, top expert = EXP_001


## 3. 소부장 use-case demo — 기술문제로 전문가 찾기 (결과)

This is the question the work set out to answer. Pick a real 소재/부품/장비
(materials / parts / equipment) SME problem, show the technology the ontology
says it needs, and the SPARQL-ranked experts — each with the *specific
overlapping skills* that put them on the list (explainable by construction).

In [5]:
SOOBUJANG = {'materials', 'parts', 'equipment'}

demo_pid = next(
    (p['problem_id'] for p in problems
     if p.get('company_type') in SOOBUJANG
     and p['problem_id'].startswith('PROB_')
     and problem_needs(p['problem_id'])),
    _demo_pid,
)
p = by_pid[demo_pid]

print(f'Problem {demo_pid}  [{p.get("company_type")} SME, '
      f'{p.get("compliance_sensitivity")}, client={p.get("client_country") or p.get("region")}]')
print(f'  title : {p.get("problem_title")}')
desc = (p.get('problem_description') or '')
print(f'  desc  : {desc[:220]}{"..." if len(desc) > 220 else ""}')

print('\nOntology says this problem needs:')
for nid, lab in sorted(problem_needs(demo_pid)):
    print(f'  • {lab}  ({nid})')

# Per-expert: which need-nodes they cover (the explanation).
EXPLAIN = PFX + '''
SELECT ?need ?lab WHERE {
  { ?prob ont:requiresSkill ?need }
  UNION { ?prob ont:involvesProcess ?p . ?p ont:requiresSkill ?need }
  UNION { ?prob ont:involvesProcess ?need }
  UNION { ?prob ont:exhibitsFailureMode ?fm . ?fm ont:isDueTo ?rc .
          ?rc ont:mitigatedBy ?m . ?m ont:mitigationProvidesSkill ?need }
  { ?ex ont:hasSkill ?need } UNION { ?ex ont:hasProcessExpertise ?need }
  OPTIONAL { ?need <http://www.w3.org/2004/02/skos/core#prefLabel> ?lab }
}'''

print(f'\nTop-10 experts for {demo_pid} (SPARQL graph overlap):')
for eid in rank_experts(demo_pid)[:10]:
    covers = {str(r.lab) if r.lab else str(r.need).rsplit("/", 1)[-1]
              for r in g.query(EXPLAIN, initBindings={
                  'prob': _uri(demo_pid), 'ex': URIRef(f'{DATA}expert/{eid}')})}
    if not covers:
        break  # tail (zero-overlap) reached
    print(f'  {eid}  {expert_label.get(eid, ""):10s} '
          f'overlap={len(covers)}  via {sorted(covers)}')

Problem PROB_001  [materials SME, confidential, client=KR]
  title : Need to optimize deposition process for cost reduction
  desc  : We need to optimize our deposition process to achieve lower defects. Current performance: 71%. Target specification: 92%. Key constraints include time constraints. Previous optimization attempts using literature review s...

Ontology says this problem needs:
  • Deposition  (process:deposition)
  • Film Stress Control  (skill:film_stress_control)
  • CVD  (subprocess:cvd)

Top-10 experts for PROB_001 (SPARQL graph overlap):


  EXP_001  김태훈        overlap=2  via ['CVD', 'Deposition']
  EXP_002  박지현        overlap=2  via ['CVD', 'Deposition']
  EXP_005  김영수        overlap=2  via ['CVD', 'Deposition']
  EXP_007  임소영        overlap=2  via ['CVD', 'Deposition']
  EXP_020  한예원        overlap=2  via ['CVD', 'Deposition']
  EXP_032  김서연        overlap=2  via ['CVD', 'Deposition']
  EXP_081  전민재        overlap=2  via ['CVD', 'Deposition']
  EXP_082  조민수        overlap=2  via ['CVD', 'Deposition']
  EXP_086  신동욱        overlap=2  via ['CVD', 'Deposition']
  EXP_095  문예린        overlap=2  via ['CVD', 'Deposition']


## 4. Same ground truth, same metrics — vs the TF-IDF floor

Ground truth and metric functions are **lifted verbatim from notebook 01** so
this is a controlled comparison of *retrieval mechanism* (ontology traversal vs
TF-IDF), holding the evaluation fixed:

- **Relevance GT.** A `(problem, expert)` pair is relevant when **≥ 2 of 3
  raters scored ≥ 2** on the **0–3 Likert** scale. NDCG uses graded relevance
  = mean rating / 3.
- **Metrics.** MRR, NDCG@5, Precision@5, Recall@10 — same code as 01.
- Reported twice: over **all rated problems** (apples-to-apples with 01) and
  over the **graph-rankable subset** (problems the lossy lift actually gave
  ≥1 need-node — the honest ceiling of this mechanism today).

In [ ]:
# ---- metric suite: single source (scripts/sdkb_nb.py; verbatim from 01) ----
from sdkb_nb import (calculate_mrr_with_ground_truth,
                     calculate_precision_at_k,
                     calculate_recall_at_k,
                     calculate_ndcg_at_k)

# ---- relevance GT: identical construction to notebook 01 ----
_consensus = (
    ratings.assign(_pos=(ratings['relevance_score'] >= 2).astype(int))
    .groupby(['problem_id', 'expert_id'])
    .agg(_n=('relevance_score', 'count'), _pos=('_pos', 'sum'),
         _mean=('relevance_score', 'mean'))
    .reset_index()
)
_consensus['is_relevant'] = _consensus['_pos'] >= 2
ground_truth_rankings = {
    pid: grp.loc[grp['is_relevant'], 'expert_id'].tolist()
    for pid, grp in _consensus.groupby('problem_id')
}
rel_scores_by_problem = {
    pid: dict(zip(grp['expert_id'], grp['_mean'] / 3.0))
    for pid, grp in _consensus.groupby('problem_id')
}

# ---- SPARQL-ranked rows for the same problem set ----
ranked_rows = [[p['problem_id']] + rank_experts(p['problem_id'])
               for p in problems]
graph_rankable = {
    p['problem_id'] for p in problems if problem_needs(p['problem_id'])
}
print(f'evaluated {len(ranked_rows)} problems; '
      f'graph-rankable (>=1 need-node): {len(graph_rankable)}/{len(ranked_rows)}')

In [7]:
def score_block(rows):
    mrr = calculate_mrr_with_ground_truth(rows, ground_truth_rankings)
    nd, p5, r10 = [], [], []
    for row in rows:
        pid, ranked = row[0], row[1:]
        rel = set(ground_truth_rankings.get(pid, []))
        if not rel:
            continue
        rmap = rel_scores_by_problem.get(pid, {e: 1.0 for e in rel})
        nd.append(calculate_ndcg_at_k(ranked, rmap, k=5))
        p5.append(calculate_precision_at_k(ranked, rel, k=5))
        r10.append(calculate_recall_at_k(ranked, rel, k=10))
    return {
        'MRR': mrr,
        'NDCG@5': float(np.mean(nd)) if nd else 0.0,
        'P@5': float(np.mean(p5)) if p5 else 0.0,
        'R@10': float(np.mean(r10)) if r10 else 0.0,
        'n_eval': len(nd),
    }


sparql_all = score_block(ranked_rows)
sparql_rk = score_block([r for r in ranked_rows if r[0] in graph_rankable])

# Notebook 01 floor, as printed by its executed §5 (re-run 01 to refresh).
FLOOR_01 = {'MRR': 0.2461, 'NDCG@5': 0.2508, 'P@5': 0.1345,
            'R@10': 0.2817, 'n_eval': 55}

rows = pd.DataFrame({
    '01 floor (TF-IDF, ref)': FLOOR_01,
    '06 SPARQL (all rated)': sparql_all,
    '06 SPARQL (graph-rankable)': sparql_rk,
}).T
rows = rows[['MRR', 'NDCG@5', 'P@5', 'R@10', 'n_eval']]
rows[['MRR', 'NDCG@5', 'P@5', 'R@10']] = \
    rows[['MRR', 'NDCG@5', 'P@5', 'R@10']].astype(float).round(4)
rows['n_eval'] = rows['n_eval'].astype(int)
print('=== Ranking quality — same GT, same metrics ===')
print(rows.to_string())
print('\nFleiss κ = 0.258 / ICC(2,1) = 0.552 (moderate) — treat single-decimal')
print('differences as rater noise, not signal (per notebook 01 Notes).')

=== Ranking quality — same GT, same metrics ===
                               MRR  NDCG@5     P@5    R@10  n_eval
01 floor (TF-IDF, ref)      0.2461  0.2508  0.1345  0.2817      55
06 SPARQL (all rated)       0.3709  0.4614  0.2545  0.4019      55
06 SPARQL (graph-rankable)  0.3736  0.5781  0.3122  0.4639      41

Fleiss κ = 0.258 / ICC(2,1) = 0.552 (moderate) — treat single-decimal
differences as rater noise, not signal (per notebook 01 Notes).


## Notes

- **What this demonstrates.** Expert retrieval can run as *graph traversal* over
  the SDKB ontology end-to-end: `make abox` lifts experts and SME problems into
  an RDF A-Box bound to the same node URIs as the domain KG, and a single
  parametrized SPARQL query answers “which experts hold the technology this
  소부장 problem needs?” — with a built-in explanation (the overlapping
  skill/process nodes), which the TF-IDF floor cannot give.
- **The lift is deterministic and lossy — reported, not hidden.**
  `scripts/build_abox_experts_problems.py` uses Tier-1 exact lexicon
  (canonical_name / id / synonyms incl. Korean) + Tier-2 curated aliases
  (`mappings/abox_term_aliases.json`). Residual loss is in
  `data/reports/abox_linking_report.json`: see `term_resolution` and
  `top_unmatched`. Genuinely out-of-ontology tags (DRAM, DDR5, FEOL, generic
  `materials`) stay unmatched **by design**. Problems with zero need-nodes are
  graph-unrankable and score ~0 — that is why the table reports both the
  all-rated and the graph-rankable columns.
- **Comparison caveat.** 01 and 06 share GT and metric code, so the columns are
  comparable as *mechanisms*, but 01 ranks all 110 experts by a dense textual
  signal whereas 06's signal is sparse integer overlap with a long zero tail —
  expect different precision/recall shapes, and read both against the κ/ICC
  noise floor.
- **Next steps to lift 06 above its lift ceiling.** (a) grow
  `abox_term_aliases.json` from `report.top_unmatched` (highest-frequency
  first); (b) link expert tags to `Material` / `Equipment` more aggressively;
  (c) richer SPARQL paths (incompatibility / not-allowed-with constraints,
  weighted edges via `ont:confidence`); (d) hybrid: graph-overlap as a
  re-ranker over the 01 candidate pool. None of these change the evaluation —
  only the mechanism — so improvements remain directly attributable.